# How to: First steps in FELiCS 

[Link to installation guide]
How to import FELiCS...

In [2]:
The <a href="https://git.tu-berlin.de/laboratory-for-flow-instabilities-and-dynamics/felics2.0/-/blob/main/DOCUMENTATION/installation_guide.md"> installation guide </a> explains how to create the conda environment, download FELiCS and install FELiCS as a package.

After installing FELiCS you can start writing your first FELiCS script.

SyntaxError: invalid syntax (1676918533.py, line 1)

We use gmsh to create a mesh. [TODO: link to tutorial]
First we create a FELiCSMesh object from a gmsh mesh file. The FELiCSMesh is a wrapper for a dolfinx mesh, that contains additional attributes and methods, e.g. the coordinate system of your case.


In [ ]:
from FELiCS.SpaceDisc.FELiCSMesh import FELiCSMesh

meshFileName = "./square_mesh.msh"
coordinateSystem = "Cartesian"

# Create a FELiCS mesh from the gmsh file
mesh = FELiCSMesh(coordinateSystem=coordinateSystem, meshFileName=meshFileName)

Info     | FELiCSMesh.py          | __init__                   (line 28  ) : Opening mesh file: ./square_mesh.msh
Info     | FELiCSMesh.py          | __init__                   (line 36  ) : Mesh contains 513 nodes and 1024 elements


Next we create a FEM function space from our FELiCSMesh object.

In [14]:
from FELiCS.SpaceDisc.FEMSpaces import getFELiCSSpace

space = getFELiCSSpace(mesh, order=2, dim=1)



Now things are getting interesting. The Field object can store fields on our function space, as the name suggests. We can import data to our Field from a h5 file, but also export our created data to h5. Fields can be naturally added to eachother using the standard python operators. Make sure, that the fields are defined on the same space, otherwise the fields cannot be added together.

In [15]:
from FELiCS.Fields.Field import Field

Field1 = Field(space, mesh=mesh)
Field1.importH5("function_values_quadratic")
Field2 = Field(space, mesh=mesh)
Field2.importH5("function_values_sine")

Field3 = Field1 + Field2

Field3.exportH5("field_sum")


In [ ]:
import numpy as np

# Define some test functions to evaluate on the mesh
def test_function_1(x):
    """Simple quadratic function: f(x,y) = x^2 + y^2"""
    return x[0]**2 + x[1]**2

def test_function_2(x):
    """Sine wave function: f(x,y) = sin(2*pi*x) * cos(2*pi*y)"""
    return np.sin(2 * np.pi * x[0]) * np.cos(2 * np.pi * x[1])

def test_function_3(x):
    """Gaussian function: f(x,y) = exp(-(x-0.5)^2 - (y-0.5)^2)"""
    return np.exp(-((x[0] - 0.5)**2 + (x[1] - 0.5)**2))

# Get coordinates of all DOFs in the P2 space
dof_coordinates = space.tabulate_dof_coordinates()
print(f"Number of DOFs: {len(dof_coordinates)}")
print(f"DOF coordinates shape: {dof_coordinates.shape}")

# Calculate function values at DOF coordinates
function_values_1 = np.zeros(len(dof_coordinates))
function_values_2 = np.zeros(len(dof_coordinates))

for i, coord in enumerate(dof_coordinates):
    function_values_1[i] = test_function_1(coord)
    function_values_2[i] = test_function_2(coord)

# Save to numpy files
np.save('function_values_quadratic.npy', function_values_1)
np.save('function_values_sine.npy', function_values_2)

print("Function values saved to .npy files:")
print("- function_values_quadratic.npy")
print("- function_values_sine.npy") 
print("- function_values_gaussian.npy")
print("- dof_coordinates.npy")


Number of DOFs: 1969
DOF coordinates shape: (1969, 3)
Function values saved to .npy files:
- function_values_quadratic.npy
- function_values_sine.npy
- function_values_gaussian.npy
- dof_coordinates.npy
